In [ ]:
import pandas as pd
import os
import gc
import functions

### Training dataset featurization

In [ ]:
def featurization (df, elem_prop='oliynyk', save = (False, "Output_featurized data.scv")):
    df = df.copy()
    df.columns = ['formula', 'target']

    # Run featurization
    X, y, formulas, skipped = functions.generate_features(
        df,
        elem_prop,     # or 'magpie', 'jarvis', etc.
        drop_duplicates=False,
        extend_features=False,
        sum_feat=False,
        mini=False
    )

    if skipped:
        skipped_formulas = pd.DataFrame(skipped, columns=["skipped"])
        skipped_formulas.to_csv("skipped_formulas.csv", index=False)
        del skipped, skipped_formulas
        gc.collect()

    # Reformat output
    output_df = pd.concat([formulas, y, X], axis=1)
    output_df = output_df.rename(columns={output_df.columns[0]: "Structure", output_df.columns[1]: "bandgap"})

    # Save to output.csv
    if save[0] == True:
        output_df.to_csv(f"{save[1]}", index=False)
        print(f"Updated file saved to: {save[1]}")

    return output_df

df = pd.read_csv("Initial Dataset.csv")
Featurized_Data = featurization(df, elem_prop='oliynyk', save = (True, "Featurized_Data.csv"))

### Feature Selection

In [ ]:
Featurized_Data = pd.read_csv("Featurized_Data.csv")
Feature_Selection_Results = functions.feature_selection_pipeline(Featurized_Data, params=None, save = (True, "Feature_Selection.csv"))

### Candidate Structures featurization

In [ ]:
def featurization (df, elem_prop='oliynyk', save = (False, "Output_featurized data.scv")):
    """input file as csv with two columns (usually), as ('formula', 'target')
    output file gives featurized vectors as ('formula', 'target', 'feature1', 'feature2', ...)
    elem_prop = [jarvis, magpie, mat2vec, oliynyk, onehot, random_200]
    if "extend_features=True" use from 3rd column to extend featureset """

    df = df.copy()
    df.columns = ['formula', 'target']

    # Run featurization
    X, y, formulas, skipped = functions.generate_features(
        df,
        elem_prop,     # or 'magpie', 'jarvis', etc.
        drop_duplicates=False,
        extend_features=False,
        sum_feat=False,
        mini=False,
        req_features= "selected_features.csv"
    )

    if skipped:
        skipped_formulas = pd.DataFrame(skipped, columns=["skipped"])
        skipped_formulas.to_csv("skipped_formulas.csv", index=False)
        del skipped, skipped_formulas
        gc.collect()

    # Reformat output
    output_df = pd.concat([formulas, y, X], axis=1)
    output_df = output_df.rename(columns={output_df.columns[0]: "Structure", output_df.columns[1]: "bandgap"})

    # Save to output.csv
    if save[0] == True:
        output_df.to_csv(f"{save[1]}", index=False)
        print(f"Updated file saved to: {save[1]}")

    return output_df

input_file = "Candidate_Structure.csv"
chunk_size = 100000
chunk_files = []
i = 0

for chunk in pd.read_csv(input_file, chunksize=chunk_size):
    chunk = chunk.iloc[:, [0]]
    chunk.insert(1, 'BandGap', 0)
    Featurized_Data = featurization(
        chunk,
        elem_prop='oliynyk',
        save=(False, "")
    )
    chunk_filename = f"chunk_{i}.csv"
    Featurized_Data.to_csv(chunk_filename, index=False)
    chunk_files.append(chunk_filename)
    i += 1

output_file = "Candidate_Structure_featurized.csv"
first = True
for file in chunk_files:
    df_chunk = pd.read_csv(file)
    df_chunk.to_csv(
        output_file,
        mode='a',
        header=first,
        index=False
    )
    first = False

for file in chunk_files:
    os.remove(file)


### Training Classification model 1

In [ ]:
# Load data
df = pd.read_csv("Filtered_Features.csv")
        

functions.train_classification_model(
    df, 
    bandgap_threshold=0.5,
    test_size=0.8,
    output_dir='results/0.5'
)

### Training Classification model 2

In [ ]:
# Load data
df = pd.read_csv("Filtered_Features.csv")

functions.train_classification_model(
    df, 
    bandgap_threshold=2.0,
    test_size=0.8,
    output_dir='results/2.0'
)

### Classification model inference 1

In [ ]:
functions.classify_structures(
    input_file="Candidate_Structure_featurized.csv",
    model_path="results/0.5/best_model.pkl",
    output_file="classified_output_0.5.csv"
)

### Classification model inference 2

In [ ]:
functions.classify_structures(
    input_file="Candidate_Structure_featurized.csv",
    model_path="results/2.0/best_model.pkl",
    output_file="classified_output_2.0.csv"
)

### Training Regression models (individual)

In [ ]:
BANDGAP_MIN = 0.15
BANDGAP_MAX = 4.0
TEST_SIZE = 0.5
N_TRIALS = 200
RANDOM_STATE = 42

functions.train_individual_models("Filtered_Features.csv", BANDGAP_MIN, BANDGAP_MAX, TEST_SIZE, N_TRIALS, RANDOM_STATE)

### Training Regression models (ensemble)

In [ ]:
BANDGAP_MIN = 0.15
BANDGAP_MAX = 4.0
RANDOM_STATE = 42
INPUT_DIR = f'regression_outputs_{BANDGAP_MIN}-{BANDGAP_MAX}'
NUMBER_OF_MODELS_LIST = [2,3,4,5]

functions.train_ensemble_models("Filtered_Features.csv", INPUT_DIR, BANDGAP_MIN, BANDGAP_MAX, NUMBER_OF_MODELS_LIST, RANDOM_STATE)

### Regression model inference

In [ ]:
BANDGAP_MIN = 0.15
BANDGAP_MAX = 4.0
INPUT_DIR = f'regression_outputs_{BANDGAP_MIN}-{BANDGAP_MAX}'
INPUT_CSV = 'Candidate_Structure_featurized.csv'
BEST_MODEL_FILE = 'Weighted_Voting_n2_model.joblib'
OUTPUT_CSV = f"Candidate_Structure_BGpredicted.csv"

functions.predict_Bandgap(INPUT_DIR, INPUT_CSV, BEST_MODEL_FILE, OUTPUT_CSV)